In [2]:
# ---------- OOV (% de tokens que acabarían como <UNK>) según el cutoff del vocabulario ----------
from collections import Counter
import pandas as pd

DATA_DIR = "../data/interim"   # ajustar la ruta si se ejecuta en Colab


def flatten(tokens):
    """Une los tokens de todas las frases de un prompt en una sola lista."""
    return [t for sentence in tokens for t in sentence]


# Tokens aplanados de cada split (se mantiene la puntuación y las mayúsculas)
docs = {
    n: pd.read_json(f"{DATA_DIR}/{n}_tok.jsonl", lines=True)["tokens"].map(flatten)
    for n in ["train", "val", "test"]
}

# El vocabulario se construye SOLO con train
train_counts = Counter(t for doc in docs["train"] for t in doc)

rows = []
for cutoff in (1, 2):
    vocab = {t for t, c in train_counts.items() if c >= cutoff}
    row = {"cutoff": cutoff, "tamaño vocab (sin especiales)": len(vocab)}
    for split, split_docs in docs.items():
        total = sum(len(doc) for doc in split_docs)
        unk = sum(t not in vocab for doc in split_docs for t in doc)
        row[f"% UNK {split}"] = 100 * unk / total
    rows.append(row)

df_oov = pd.DataFrame(rows).set_index("cutoff")
# % UNK train: con cutoff=1 es 0 (el <UNK> no se entrena nunca); con cutoff=2 es el
# porcentaje de tokens de train con el que la red aprende a gestionar <UNK>.
df_oov.round(3)


,tamaño vocab (sin especiales),% UNK train,% UNK val,% UNK test
cutoff,,,,
1,82469,0.000,2.343,2.131
2,46632,1.068,3.002,2.786


Cosas a tener en cuenta: 

UNK: para palabras no en el vocaulario



W2v normal vs W2v normal finetuneado vs W2v propio vs Elmo




1: Crear el vocabularuio: mapear tokens de interim/test,trainy,val en jsonl en columna tokens A Numeros
2: Wrapper: añadimso marcadores a la frase: unk, mask, begin y end 
3: Padding

4.CNN
- OJO que funcion de activacion emtemos: Elu(collab) o Relu (recordar la funcion de meter no linealidad que tiene la funcion de activacion)
- Decidir el numero de capas de la red neuronal. En base a que?
--Entender los shapes de cada capa

- Decididr y entender el valor y el signficado de estos argumentos: 
            embedding_size (int): size of the embedding vectors
            num_embeddings (int): number of embedding vectors
            filter_width (int): width of the convolutional kernels
            num_channels (int): number of convolutional kernels per layer
            hidden_dim (int): the size of the hidden dimension
            num_classes (int): the number of classes in classification
            dropout_p (float): a dropout parameter
            pretrained_embeddings (numpy.array): previously trained word embeddings
                default is None. If provided,
            padding_idx (int): an index representing a null position


- Capa final sigmoid o softmax y porque? se supone que sigmoi e spioirque solo necesita 1 neurona final y softmax varias. 


In [ ]:
import os
import json
import random
import time
from argparse import Namespace
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from IPython.display import display
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             precision_score, recall_score)


def flatten(tokens):
    """Une los tokens de todas las frases de un prompt en una sola lista."""
    return [t for sentence in tokens for t in sentence]

## Vocabulary


In [ ]:
class Vocabulary(object):
    """Clase para procesar texto y extrar el vocabulario existente para su posterior mapeo."""

    def __init__(self, token_to_idx=None):
        """
        Args:
            token_to_idx (dict): a pre-existing map of tokens to indices
        """

        if token_to_idx is None:
            token_to_idx = {}
        self._token_to_idx = token_to_idx

        self._idx_to_token = {idx: token
                              for token, idx in self._token_to_idx.items()}

    def to_serializable(self):
        """ returns a dictionary that can be serialized """
        return {'token_to_idx': self._token_to_idx}

    @classmethod
    def from_serializable(cls, contents):
        """ instantiates the Vocabulary from a serialized dictionary """
        return cls(**contents)

    def add_token(self, token):
        """Update mapping dicts based on the token.

        Args:
            token (str): the item to add into the Vocabulary
        Returns:
            index (int): the integer corresponding to the token
        """
        if token in self._token_to_idx:
            index = self._token_to_idx[token]
        else:
            index = len(self._token_to_idx)
            self._token_to_idx[token] = index
            self._idx_to_token[index] = token
        return index

    def add_many(self, tokens):
        """Add a list of tokens into the Vocabulary

        Args:
            tokens (list): a list of string tokens
        Returns:
            indices (list): a list of indices corresponding to the tokens
        """
        return [self.add_token(token) for token in tokens]

    def lookup_token(self, token):
        """Retrieve the index associated with the token

        Args:
            token (str): the token to look up
        Returns:
            index (int): the index corresponding to the token
        """
        return self._token_to_idx[token]

    def lookup_index(self, index):
        """Return the token associated with the index

        Args:
            index (int): the index to look up
        Returns:
            token (str): the token corresponding to the index
        Raises:
            KeyError: if the index is not in the Vocabulary
        """
        if index not in self._idx_to_token:
            raise KeyError("the index (%d) is not in the Vocabulary" % index)
        return self._idx_to_token[index]

    def __str__(self):
        return "<Vocabulary(size=%d)>" % len(self)

    def __len__(self):
        return len(self._token_to_idx)

## Class to add especial tokens to the vocabulary
- < UNK >: unknown tokens
- < MASK >: padding
- < BEGIN-OF-SEQUENCE >
- < END-OF-SEQUENCE >

In [ ]:
class SequenceVocabulary(Vocabulary):
    """Extensión de la clase vocabulario para lidiar con la secuencialidad del texto."""
    def __init__(self, token_to_idx=None, unk_token="<UNK>",
                 mask_token="<MASK>", begin_seq_token="<BEGIN>",
                 end_seq_token="<END>"):

        super(SequenceVocabulary, self).__init__(token_to_idx) #Llamamos al constructor padre.

        self._mask_token = mask_token
        self._unk_token = unk_token
        self._begin_seq_token = begin_seq_token
        self._end_seq_token = end_seq_token

        #Usamos el vocabulario inicial y añadimos los nuevos tokens de uso especial.
        self.mask_index = self.add_token(self._mask_token) #Padding
        self.unk_index = self.add_token(self._unk_token) #Tokens desconocidos
        self.begin_seq_index = self.add_token(self._begin_seq_token) #Inicio de una frase.
        self.end_seq_index = self.add_token(self._end_seq_token) #Fin de una frase.

    def to_serializable(self):
        contents = super(SequenceVocabulary, self).to_serializable()
        contents.update({'unk_token': self._unk_token,
                         'mask_token': self._mask_token,
                         'begin_seq_token': self._begin_seq_token,
                         'end_seq_token': self._end_seq_token})
        return contents

    def lookup_token(self, token):
        """Retrieve the index associated with the token
          or the UNK index if token isn't present.

        Args:
            token (str): the token to look up
        Returns:
            index (int): the index corresponding to the token
        Notes:
            `unk_index` needs to be >=0 (having been added into the Vocabulary)
              for the UNK functionality
        """
        if self.unk_index >= 0:
            return self._token_to_idx.get(token, self.unk_index)
        else:
            return self._token_to_idx[token]

## Vectorizer
- Pasa los tokens a su id del vocabulario y añade los tokens epspeciales cuando haga falta. 

In [ ]:
def truncate_head_tail(tokens, max_tokens, head_ratio=0.5):
    """Si el prompt tiene más de max_tokens tokens, conserva el principio (cabeza) y el
    final (cola) y descarta la parte central."""
    if len(tokens) <= max_tokens:
        return tokens
    head = int(max_tokens * head_ratio)
    tail = max_tokens - head
    return tokens[:head] + tokens[len(tokens) - tail:]


class PromptVectorizer(object):
    """Usando el vocabulario, transforma los tokens de un prompt a sus ids numéricos y añade
    los tokens auxiliares (BEGIN, END y MASK para el padding)."""
    def __init__(self, vocab):
        self.vocab = vocab  # Vocabulario de todos los tokens que usamos (SequenceVocabulary).

    def vectorize(self, tokens, vector_length):
        """
        Args:
            tokens (list[str]): tokens del prompt (ya aplanados)
            vector_length (int): longitud final del vector, contando <BEGIN> y <END>
        Returns:
            np.array de ids (int64) de longitud vector_length
        """
        # Primero se trunca (cabeza y cola) para que <END> nunca se pierda
        tokens = truncate_head_tail(tokens, vector_length - 2)

        indices = [self.vocab.begin_seq_index]
        indices.extend(self.vocab.lookup_token(token) for token in tokens)
        indices.append(self.vocab.end_seq_index)

        out_vector = np.full(vector_length, self.vocab.mask_index, dtype=np.int64)
        out_vector[:len(indices)] = indices
        return out_vector

    @classmethod
    def from_dataframe(cls, df, cutoff=2):
        """Crea el vocabulario con los tokens que aparecen al menos `cutoff` veces.
        Se mantiene la puntuación (igual que en el Word2Vec). Usar SOLO el train."""
        word_counts = Counter(t for tokens in df.tokens_flat for t in tokens)

        vocab = SequenceVocabulary()
        for word, word_count in word_counts.items():
            if word_count >= cutoff:
                vocab.add_token(word)

        return cls(vocab)

    @classmethod
    def from_serializable(cls, contents):
        return cls(SequenceVocabulary.from_serializable(contents['vocab']))

    def to_serializable(self):
        return {'vocab': self.vocab.to_serializable()}


def generate_batches(dataset, batch_size, shuffle=True,
                     drop_last=True, device="cpu"):
    """
    A generator function which wraps the PyTorch DataLoader. It will
      ensure each tensor is on the write device location.
    """
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size,
                            shuffle=shuffle, drop_last=drop_last)

    for data_dict in dataloader:
        out_data_dict = {}
        for name, tensor in data_dict.items():
            out_data_dict[name] = data_dict[name].to(device)
        yield out_data_dict

## Usar el vectorizer usando el dataset

In [ ]:
class PromptDataset(Dataset):
    """Dataset de prompts ya tokenizados. Vectoriza todos los prompts una sola vez
    (ids con <BEGIN>, <END> y padding) y los guarda como tensores."""
    def __init__(self, df, vectorizer, max_tokens=1022):
        """
        Args:
            df (pandas.DataFrame): columnas `tokens_flat`, `jailbreak` y `split`
            vectorizer (PromptVectorizer): vectorizer creado solo con train
            max_tokens (int): tokens máximos por prompt (sin contar <BEGIN> y <END>)
        """
        self._vectorizer = vectorizer
        self._max_seq_length = max_tokens + 2  # + <BEGIN> y <END>

        self._data = {}
        for split in ["train", "val", "test"]:
            sub = df[df.split == split]
            X = np.stack([vectorizer.vectorize(tokens, self._max_seq_length)
                          for tokens in sub.tokens_flat])
            y = sub.jailbreak.to_numpy(dtype=np.float32)   # etiqueta binaria como float (para BCE)
            self._data[split] = (torch.from_numpy(X), torch.from_numpy(y))

        # pos_weight para el desbalanceo: nº negativos / nº positivos (solo train)
        y_train = self._data["train"][1]
        n_pos = int(y_train.sum().item())
        n_neg = len(y_train) - n_pos
        self.pos_weight = n_neg / n_pos

        self.set_split("train")

    @classmethod
    def from_dataframe(cls, df, cutoff=2, max_tokens=1022):
        """Crea el vectorizer (vocabulario solo con train) y el dataset."""
        vectorizer = PromptVectorizer.from_dataframe(df[df.split == "train"], cutoff=cutoff)
        return cls(df, vectorizer, max_tokens=max_tokens)

    def save_vectorizer(self, vectorizer_filepath):
        """saves the vectorizer to disk using json"""
        with open(vectorizer_filepath, "w") as fp:
            json.dump(self._vectorizer.to_serializable(), fp)

    def get_vectorizer(self):
        """ returns the vectorizer """
        return self._vectorizer

    def set_split(self, split="train"):
        """ selecciona el split activo: train, val o test """
        self._target_split = split
        self._X, self._y = self._data[split]

    def __len__(self):
        return len(self._y)

    def __getitem__(self, index):
        """Devuelve el vector de ids del prompt (x_data) y su etiqueta 0/1 (y_target)."""
        return {'x_data': self._X[index],
                'y_target': self._y[index]}

    def get_num_batches(self, batch_size):
        return len(self) // batch_size

## Early stopping y Checkpoints del modelo

In [ ]:
def make_train_state(args, model_filename):
    return {'stop_early': False,
            'early_stopping_step': 0,
            'best_val_mcc': -1.0,       # el checkpoint se guarda cuando mejora el MCC de validación
            'best_epoch': -1,
            'epoch_index': 0,
            'train_loss': [],
            'train_mcc': [],
            'val_loss': [],
            'val_mcc': [],
            'model_filename': str(model_filename)}


def update_train_state(args, model, train_state):
    """Early stopping y checkpoint según el MCC de validación (mayor es mejor).

    - Si el MCC de val mejora: se guarda el modelo y se reinicia el contador.
    - Si no mejora: el contador sube y, al llegar a early_stopping_criteria, se para.
    """
    val_mcc = train_state['val_mcc'][-1]

    if val_mcc > train_state['best_val_mcc']:
        torch.save(model.state_dict(), train_state['model_filename'])
        train_state['best_val_mcc'] = val_mcc
        train_state['best_epoch'] = train_state['epoch_index']
        train_state['early_stopping_step'] = 0
    else:
        train_state['early_stopping_step'] += 1

    train_state['stop_early'] = \
        train_state['early_stopping_step'] >= args.early_stopping_criteria
    return train_state


def compute_metrics(y_true, y_prob, threshold=0.5):
    """Mismas métricas que en shallow learning, calculadas sobre TODO el conjunto."""
    y_pred = (y_prob >= threshold).astype(int)
    return {"F1": f1_score(y_true, y_pred, zero_division=0),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "MCC": matthews_corrcoef(y_true, y_pred)}


def evaluate(model, dataset, split, loss_func, batch_size, device):
    """Evalúa TODO el split: sin mezclar (shuffle=False) y sin descartar el último
    batch (drop_last=False). Devuelve (loss media, etiquetas, probabilidades)."""
    dataset.set_split(split)
    model.eval()
    total_loss, ys, probs = 0.0, [], []
    with torch.no_grad():
        for batch in generate_batches(dataset, batch_size, shuffle=False,
                                      drop_last=False, device=device):
            logits = model(batch['x_data'])
            total_loss += loss_func(logits, batch['y_target']).item() * len(batch['y_target'])
            ys.append(batch['y_target'].cpu())
            probs.append(torch.sigmoid(logits).cpu())
    y_true = torch.cat(ys).numpy().astype(int)
    y_prob = torch.cat(probs).numpy()
    return total_loss / len(y_true), y_true, y_prob

## Implementación CNN para text classification

In [ ]:
class CNN_NLP(nn.Module):
    """An 1D Convulational Neural Network for Sentence Classification.
    Adapted from: https://www.kaggle.com/code/williamlwcliu/cnn-text-classification-pytorch"""
    def __init__(self,
                 pretrained_embedding=None,
                 freeze_embedding=False,
                 vocab_size=None,
                 embed_dim=100,
                 filter_sizes=[3, 4, 5],
                 num_filters=[100, 100, 100],
                 num_classes=1,
                 dropout=0.5,
                 use_embedding_layer=True):
        """
        The constructor for CNN_NLP class.

        Args:
            pretrained_embedding (numpy.array): Pretrained embeddings with
                shape (vocab_size, embed_dim)
            freeze_embedding (bool): True congela la capa Embedding (no se entrena);
                False la ajusta durante el entrenamiento (fine-tuning).
            vocab_size (int): Need to be specified when not pretrained word
                embeddings are not used.
            embed_dim (int): Dimension of word vectors (la fija el embedding usado).
            filter_sizes (List[int]): List of filter sizes. Default: [3, 4, 5]
            num_filters (List[int]): List of number of filters, has the same
                length as `filter_sizes`. Default: [100, 100, 100]
            num_classes (int): 1 = clasificación binaria (una neurona, logit para BCE).
            dropout (float): Dropout rate. Default: 0.5
            use_embedding_layer (bool): False si la entrada ya son vectores
                (batch, longitud, embed_dim), como los de ELMo.
        """

        super(CNN_NLP, self).__init__()
        self.num_classes = num_classes
        # Embedding layer
        if not use_embedding_layer:
            self.emb = None
        elif pretrained_embedding is not None:
            self.emb = nn.Embedding.from_pretrained(
                torch.from_numpy(pretrained_embedding).float(),
                freeze=freeze_embedding,
                padding_idx=0)
        else:
            self.emb = nn.Embedding(num_embeddings=vocab_size,
                                    embedding_dim=embed_dim,
                                    padding_idx=0,
                                    max_norm=5.0)
        # Conv Network
        self.conv1d_list = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim,
                      out_channels=num_filters[i],
                      kernel_size=filter_sizes[i])
            for i in range(len(filter_sizes))
        ])
        # Fully-connected layer and Dropout
        self.fc = nn.Linear(int(np.sum(num_filters)), num_classes)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, input_ids):
        """Perform a forward pass through the network.

        Args:
            input_ids (torch.Tensor): ids de tokens (batch_size, max_sent_length) o, si
                use_embedding_layer=False, vectores (batch_size, max_sent_length, embed_dim)

        Returns:
            logits (torch.Tensor): (batch_size,) si num_classes == 1
                (sin sigmoid: lo incluye BCEWithLogitsLoss), (batch_size, num_classes) si no.
        """

        # Get embeddings from `input_ids`. Output shape: (b, max_len, embed_dim)
        if self.emb is not None:
            x_embed = self.emb(input_ids).float()
        else:
            x_embed = input_ids.float()

        # Permute `x_embed` to match input shape requirement of `nn.Conv1d`.
        # Output shape: (b, embed_dim, max_len)
        x_reshaped = x_embed.permute(0, 2, 1)

        # Apply CNN and ReLU. Output shape: (b, num_filters[i], L_out)
        x_conv_list = [F.relu(conv1d(x_reshaped)) for conv1d in self.conv1d_list]

        # Max pooling. Output shape: (b, num_filters[i], 1)
        x_pool_list = [F.max_pool1d(x_conv, kernel_size=x_conv.shape[2])
            for x_conv in x_conv_list]

        # Concatenate x_pool_list to feed the fully connected layer.
        # Output shape: (b, sum(num_filters))
        x_fc = torch.cat([x_pool.squeeze(dim=2) for x_pool in x_pool_list],
                         dim=1)

        # Compute logits. Output shape: (b, num_classes)
        logits = self.fc(self.dropout(x_fc))

        if self.num_classes == 1:
            logits = logits.squeeze(dim=-1)   # (b,)

        return logits

## Utils


In [ ]:
_google_w2v = None


def set_seed_everywhere(seed, cuda):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda:
        torch.cuda.manual_seed_all(seed)


def load_google_w2v():
    """Carga (una sola vez) el Word2Vec de Google News (se descarga la primera vez)."""
    global _google_w2v
    if _google_w2v is None:
        import gensim.downloader as api
        _google_w2v = api.load("word2vec-google-news-300")
    return _google_w2v


def build_embedding_matrix(vocab, keyed_vectors, seed=1337):
    """Matriz (len(vocab), dim) en la que la fila i es el vector del token con id i
    en el vocabulario de la CNN.
      - <MASK> (padding): ceros
      - <UNK>: media de los vectores encontrados
      - <BEGIN>, <END> y tokens que no están en el embedding: aleatorios U(-0.25, 0.25) (Kim, 2014)
    """
    rng = np.random.default_rng(seed)
    dim = keyed_vectors.vector_size
    matrix = np.zeros((len(vocab), dim), dtype=np.float32)
    special = {vocab.mask_index, vocab.unk_index, vocab.begin_seq_index, vocab.end_seq_index}

    found = []
    for idx in range(len(vocab)):
        if idx in special:
            continue
        token = vocab.lookup_index(idx)
        if token in keyed_vectors.key_to_index:
            matrix[idx] = keyed_vectors[token]
            found.append(idx)
        else:
            matrix[idx] = rng.uniform(-0.25, 0.25, dim)

    matrix[vocab.unk_index] = matrix[found].mean(axis=0)
    for idx in (vocab.begin_seq_index, vocab.end_seq_index):
        matrix[idx] = rng.uniform(-0.25, 0.25, dim)

    n_words = len(vocab) - len(special)
    print(f"Tokens del vocabulario con vector en el embedding: "
          f"{len(found)}/{n_words} ({100 * len(found) / n_words:.1f}%)")
    return matrix

## Entrenamiento y evaluación de un experimento
Función reutilizable: entrena la CNN (BCE con `pos_weight`), vigila el MCC de validación (checkpoint + early stopping), registra loss y MCC en TensorBoard, prueba varios umbrales en validación y evalúa en test con el mejor umbral.

Ver los logs: `tensorboard --logdir runs` (desde la carpeta `notebooks`).

In [ ]:
all_results = {}   # resultados de cada experimento (para la tabla de comparación)


def run_experiment(exp_name, dataset, args, embedding_matrix=None, freeze_embedding=True,
                   embed_dim=None, use_embedding_layer=True):
    """Entrena y evalúa (SOLO con validación) la CNN con un embedding concreto.
    El test no se toca aquí: se evalúa una única vez al final del notebook.
    Todas las métricas y resultados se guardan en el log de TensorBoard (runs/<exp_name>/...).

    Args:
        exp_name: nombre del experimento (carpeta de TensorBoard y del checkpoint)
        dataset: PromptDataset (ids) o ElmoDataset (vectores), con atributo pos_weight
        embedding_matrix: matriz (vocabulario, dim) para la capa Embedding (None con ELMo)
        freeze_embedding: True = capa Embedding congelada; False = se ajusta (fine-tuning)
        embed_dim: dimensión del embedding (si no se pasa, sale de la matriz)
        use_embedding_layer: False si la entrada ya son vectores (ELMo)
    """
    if embed_dim is None:
        embed_dim = embedding_matrix.shape[1]      # el tamaño lo fija el embedding

    set_seed_everywhere(args.seed, args.cuda)
    device = args.device

    # TensorBoard: una carpeta por experimento (y por ejecución, para no mezclar repeticiones)
    log_dir = Path(args.log_dir) / exp_name / time.strftime("%Y%m%d-%H%M%S")
    writer = SummaryWriter(log_dir=str(log_dir))
    model_file = Path(args.model_dir) / f"{exp_name}.pth"
    model_file.parent.mkdir(parents=True, exist_ok=True)

    classifier = CNN_NLP(pretrained_embedding=embedding_matrix,
                         freeze_embedding=freeze_embedding,
                         embed_dim=embed_dim,
                         filter_sizes=args.filter_sizes,
                         num_filters=args.num_filters,
                         num_classes=1,                      # clasificación binaria: 1 neurona + BCE
                         dropout=args.dropout_p,
                         use_embedding_layer=use_embedding_layer).to(device)

    # BCE con pos_weight = nº negativos / nº positivos (desbalanceo)
    loss_func = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([dataset.pos_weight], dtype=torch.float32, device=device))
    optimizer = optim.Adam([p for p in classifier.parameters() if p.requires_grad],
                           lr=args.learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max',
                                                     factor=0.5, patience=1)  # sobre el MCC de val
    train_state = make_train_state(args, model_file)

    if not use_embedding_layer:
        emb_state = "sin capa Embedding (la entrada son los vectores)"
    else:
        emb_state = "congelado" if freeze_embedding else "entrenable (fine-tuning)"
    print(f"=== Experimento: {exp_name} | embed_dim={embed_dim} | embedding: {emb_state} | "
          f"pos_weight={dataset.pos_weight:.2f} | logs: {log_dir}")

    try:
        for epoch_index in range(args.num_epochs):
            t0 = time.time()
            train_state['epoch_index'] = epoch_index

            # ----- Entrenamiento (batches mezclados) -----
            dataset.set_split('train')
            classifier.train()
            running_loss, ys, probs = 0.0, [], []
            for batch_index, batch in enumerate(generate_batches(
                    dataset, args.batch_size, shuffle=True, drop_last=True, device=device)):
                if args.show_summary and epoch_index == 0 and batch_index == 0:
                    summary(classifier, input_data=[batch['x_data']],
                            col_names=("input_size", "output_size", "num_params"))
                optimizer.zero_grad()
                logits = classifier(batch['x_data'])
                loss = loss_func(logits, batch['y_target'])
                loss.backward()
                optimizer.step()
                running_loss += (loss.item() - running_loss) / (batch_index + 1)
                ys.append(batch['y_target'].cpu())
                probs.append(torch.sigmoid(logits).detach().cpu())

            # Métricas sobre todos los prompts de la época (no la media de métricas por batch)
            y_true = torch.cat(ys).numpy().astype(int)
            y_prob = torch.cat(probs).numpy()
            train_metrics = compute_metrics(y_true, y_prob, args.monitor_threshold)

            # ----- Validación (todo val, sin mezclar y sin descartar nada) -----
            val_loss, y_val, p_val = evaluate(classifier, dataset, 'val', loss_func,
                                              args.batch_size, device)
            val_metrics_epoch = compute_metrics(y_val, p_val, args.monitor_threshold)
            train_mcc, val_mcc = train_metrics["MCC"], val_metrics_epoch["MCC"]

            train_state['train_loss'].append(running_loss)
            train_state['train_mcc'].append(train_mcc)
            train_state['val_loss'].append(val_loss)
            train_state['val_mcc'].append(val_mcc)

            # TensorBoard: loss y TODAS las métricas (F1, accuracy, precision, recall, MCC)
            # de train y de validación, época a época (umbral = args.monitor_threshold)
            writer.add_scalar("loss/train", running_loss, epoch_index)
            writer.add_scalar("loss/val", val_loss, epoch_index)
            for metric_name, value in train_metrics.items():
                writer.add_scalar(f"{metric_name.lower()}/train", value, epoch_index)
            for metric_name, value in val_metrics_epoch.items():
                writer.add_scalar(f"{metric_name.lower()}/val", value, epoch_index)
            writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch_index)

            train_state = update_train_state(args, classifier, train_state)
            scheduler.step(val_mcc)
            print(f"Época {epoch_index:2d} | train loss {running_loss:.4f} MCC {train_mcc:.4f} | "
                  f"val loss {val_loss:.4f} MCC {val_mcc:.4f} | "
                  f"sin mejora: {train_state['early_stopping_step']} | {time.time() - t0:.0f}s")
            if train_state['stop_early']:
                print("Early stopping")
                break
    except KeyboardInterrupt:
        print("Entrenamiento interrumpido")

    # ----- Mejor checkpoint: se prueban los umbrales en validación y se elige el mejor -----
    classifier.load_state_dict(torch.load(model_file, map_location=device))
    val_loss, y_val, p_val = evaluate(classifier, dataset, 'val', loss_func,
                                      args.batch_size, device)
    df_thresholds = pd.DataFrame(
        [{"umbral": t, **compute_metrics(y_val, p_val, t)} for t in args.thresholds]
    ).set_index("umbral")
    best_threshold = df_thresholds["MCC"].idxmax()     # el umbral se elige SOLO con validación
    val_metrics = df_thresholds.loc[best_threshold].to_dict()
    best_epoch = train_state['best_epoch']

    print(f"\nMejor época: {best_epoch} | umbral elegido en val: {best_threshold}")
    print("Validación con distintos umbrales:")
    display(df_thresholds.round(4))

    # ----- TensorBoard: resultados finales de validación (persisten en el log) -----
    # Tabla de los umbrales probados (pestaña "Text")
    table = "    " + df_thresholds.round(4).to_string().replace("\n", "\n    ")
    writer.add_text("validacion/umbrales_probados", table, best_epoch)
    # Umbral elegido y métricas finales de validación con ese umbral
    writer.add_scalar("final_val/umbral_elegido", best_threshold, best_epoch)
    writer.add_scalar("final_val/mejor_epoca", best_epoch, best_epoch)
    for metric_name, value in val_metrics.items():
        writer.add_scalar(f"final_val/{metric_name.lower()}", value, best_epoch)
    # Hiperparámetros + métricas finales (pestaña "HParams", para comparar experimentos)
    writer.add_hparams(
        {"embed_dim": embed_dim,
         "embedding_congelado": bool(freeze_embedding and use_embedding_layer),
         "capa_embedding": bool(use_embedding_layer),
         "max_tokens": args.max_tokens, "cutoff": args.cutoff,
         "batch_size": args.batch_size, "learning_rate": args.learning_rate,
         "dropout": args.dropout_p, "pos_weight": float(dataset.pos_weight),
         "filter_sizes": str(args.filter_sizes), "num_filters": str(args.num_filters),
         "umbral_elegido": float(best_threshold), "mejor_epoca": int(best_epoch)},
        {f"hparam/val_{metric_name.lower()}": value for metric_name, value in val_metrics.items()},
        run_name="hparams")
    writer.close()

    result = {"exp_name": exp_name,
              "best_epoch": best_epoch,
              "threshold": best_threshold,
              "val_thresholds": df_thresholds,
              "val_metrics": val_metrics,
              "train_state": train_state,
              "log_dir": str(log_dir),          # para añadir después más resultados al mismo log
              # Para evaluar test UNA sola vez al final, con el modelo que se elija
              "model": classifier,
              "dataset": dataset,
              "loss_func": loss_func}
    all_results[exp_name] = result
    return result


In [ ]:
# Evaluación en VALIDACIÓN quitando los prompts que contienen [PROMPT] o [TARGETLANGUAGE].
# En la tokenización: "[PROMPT]" -> "<prompt>"  y  "[TARGETLANGUAGE]" -> "<targetlanguage>".
# Se usa el mejor checkpoint (el de mejor MCC en validación) y el MISMO umbral que ese experimento
# eligió en la validación completa. El test no se toca.
PLACEHOLDERS = {"<prompt>", "<targetlanguage>"}


def log_to_tensorboard(exp_name, prefix, metrics, table_text=None, step=0):
    """Añade resultados al log de TensorBoard del experimento (la misma carpeta del entrenamiento).
    Los valores salen como escalares `<prefix>/<métrica>` y la tabla como texto (pestaña Text)."""
    writer = SummaryWriter(log_dir=all_results[exp_name]["log_dir"])
    for name, value in metrics.items():
        writer.add_scalar(f"{prefix}/{name.lower()}", float(value), step)
    if table_text is not None:
        writer.add_text(f"{prefix}/tabla", table_text, step)
    writer.close()


def eval_val_sin_placeholders(exp_name):
    r = all_results[exp_name]

    # r["model"] ya es el mejor checkpoint: run_experiment lo recarga al terminar el entrenamiento
    _, y_val, p_val = evaluate(r["model"], r["dataset"], "val", r["loss_func"],
                               args.batch_size, args.device)

    # Mismo orden que las predicciones: los prompts de validación tal y como están en df
    val_tokens = df[df.split == "val"].tokens_flat
    has_placeholder = np.array([bool(PLACEHOLDERS & set(tokens)) for tokens in val_tokens])
    assert len(has_placeholder) == len(y_val), "El orden/tamaño de val no coincide con las predicciones"

    thr = r["threshold"]
    print(f"[{exp_name}] Prompts de validación eliminados por tener placeholders: "
          f"{int(has_placeholder.sum())} de {len(y_val)} "
          f"(jailbreaks entre ellos: {int(y_val[has_placeholder].sum())}) | umbral = {thr}")

    rows = []
    for name, mask in [("Validación completa", np.ones(len(y_val), dtype=bool)),
                       ("Validación sin placeholders", ~has_placeholder)]:
        rows.append({"Conjunto": name,
                     "Prompts": int(mask.sum()),
                     "Jailbreaks": int(y_val[mask].sum()),
                     **compute_metrics(y_val[mask], p_val[mask], thr)})
    df_res = pd.DataFrame(rows)
    display(df_res.round(4))

    # Guardado persistente en archivo (para no tener que volver a ejecutarlo)
    out_dir = Path(args.model_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"val_{exp_name}_sin_placeholders.json", "w", encoding="utf-8") as f:
        json.dump({"experimento": exp_name,
                   "umbral": float(thr),
                   "placeholders": sorted(PLACEHOLDERS),
                   "prompts_eliminados": int(has_placeholder.sum()),
                   "filas": [{k: (float(v) if isinstance(v, (float, np.floating)) else
                                  int(v) if isinstance(v, (int, np.integer)) else v)
                              for k, v in row.items()} for row in rows]},
                  f, indent=2, ensure_ascii=False)
    print(f"Guardado en {out_dir}/val_{exp_name}_sin_placeholders.json")

    # Guardado también en TensorBoard (métricas de la validación sin placeholders + tabla)
    sin_ph = rows[1]
    log_to_tensorboard(
        exp_name, "val_sin_placeholders",
        {**{k: v for k, v in sin_ph.items() if k not in ("Conjunto",)}, "umbral": thr},
        table_text="    " + df_res.round(4).to_string(index=False).replace("\n", "\n    "),
        step=r["best_epoch"])
    return df_res


## W2V raw

In [ ]:
args = Namespace(
    # Data and Path hyper parameters (todo en local, sin Drive)
    data_dir="../data/interim",                                      # tokens ya divididos en train / val / test
    w2v_domain_path="../data/processed/word2vec_trained_from_zero.model",
    elmo_dir="../data/models/elmo",
    elmo_cache_dir="../data/processed/elmo_cache",
    vectorizer_file="model_storage/vectorizer.json",
    model_dir="model_storage",                                       # checkpoints (uno por experimento)
    log_dir="runs",                                                  # TensorBoard: tensorboard --logdir runs
    # Preprocesado
    cutoff=2,                 # frecuencia mínima (en train) para entrar en el vocabulario
    max_tokens=1022,          # truncado cabeza + cola; con <BEGIN> y <END> son 1024
    elmo_max_tokens=128,      # ELMo es muy lento: se usa un truncado menor (ver la celda de ELMo)
    # Model hyper parameters (el tamaño del embedding lo fija cada experimento)
    filter_sizes=[3, 4, 5],
    num_filters=[100, 100, 100],
    dropout_p=0.5,
    # Training hyper parameter
    seed=1337,
    learning_rate=0.001,
    batch_size=128,
    num_epochs=30,
    early_stopping_criteria=5,
    # Umbrales que se prueban en validación (bajo, estándar y alto)
    thresholds=(0.3, 0.5, 0.7),
    monitor_threshold=0.5,    # umbral con el que se calcula el MCC para checkpoint / early stopping
    show_summary=True,        # muestra los shapes de cada capa al empezar cada experimento
    # Runtime option
    cuda=True,
)

In [ ]:
# Dispositivo y semilla
args.cuda = args.cuda and torch.cuda.is_available()
args.device = torch.device("cuda" if args.cuda else "cpu")
print("Using CUDA: {}".format(args.cuda))
set_seed_everywhere(args.seed, args.cuda)

# Datos: tokens ya divididos en interim (train / val / test), con su columna `split`
df = pd.concat(
    [pd.read_json(f"{args.data_dir}/{n}_tok.jsonl", lines=True) for n in ["train", "val", "test"]],
    ignore_index=True,
)[["tokens", "jailbreak", "split"]]
df["tokens_flat"] = df["tokens"].map(flatten)     # aplanado: una lista de tokens por prompt

# Vocabulario SOLO con train (cutoff=2) y dataset con truncado cabeza + cola
dataset = PromptDataset.from_dataframe(df, cutoff=args.cutoff, max_tokens=args.max_tokens)
Path(args.vectorizer_file).parent.mkdir(parents=True, exist_ok=True)
dataset.save_vectorizer(args.vectorizer_file)
vectorizer = dataset.get_vectorizer()
vocab = vectorizer.vocab

print("Tamaño del vocabulario (con tokens especiales):", len(vocab))
print("Prompts por split:", df.split.value_counts().to_dict())
print(f"pos_weight (nº negativos / nº positivos en train): {dataset.pos_weight:.2f}")

In [ ]:
# Matriz de embeddings con el Word2Vec de Google News (300 dim), alineada con los ids del vocabulario
google_w2v = load_google_w2v()
matrix_google = build_embedding_matrix(vocab, google_w2v)
print("Forma de la matriz de embeddings:", matrix_google.shape)

In [ ]:
# Experimento 1: Word2Vec de Google News CONGELADO (la capa Embedding no se entrena)
result_w2v_raw = run_experiment("w2v_raw", dataset, args,
                                embedding_matrix=matrix_google, freeze_embedding=True)

In [ ]:
# Validación SIN prompts con [PROMPT] / [TARGETLANGUAGE] (mejor checkpoint de este experimento)
df_val_sin_ph_w2v_raw = eval_val_sin_placeholders("w2v_raw")

In [ ]:
#Classification

## W2V Finetuned

In [ ]:
# Experimento 2: mismo Word2Vec de Google News, pero la capa Embedding se AJUSTA durante el entrenamiento
if "matrix_google" not in globals():
    matrix_google = build_embedding_matrix(vocab, load_google_w2v())

result_w2v_finetuned = run_experiment("w2v_finetuned", dataset, args,
                                      embedding_matrix=matrix_google, freeze_embedding=False)

In [ ]:
# Validación SIN prompts con [PROMPT] / [TARGETLANGUAGE] (mejor checkpoint de este experimento)
df_val_sin_ph_w2v_finetuned = eval_val_sin_placeholders("w2v_finetuned")

## W2V trained for our domain

In [ ]:
# Experimento 3: Word2Vec adaptado a nuestro dominio (notebook 03: vectores de Google News congelados
# + vectores propios entrenados para los tokens OOV). Se carga el modelo guardado en processed.
from gensim.models import Word2Vec

domain_w2v = Word2Vec.load(args.w2v_domain_path)
matrix_domain = build_embedding_matrix(vocab, domain_w2v.wv)
print("Forma de la matriz de embeddings:", matrix_domain.shape)

# freeze_embedding=True para compararlo en igualdad de condiciones con el experimento 1
result_w2v_domain = run_experiment("w2v_domain", dataset, args,
                                   embedding_matrix=matrix_domain, freeze_embedding=True)

In [ ]:
# Validación SIN prompts con [PROMPT] / [TARGETLANGUAGE] (mejor checkpoint de este experimento)
df_val_sin_ph_w2v_domain = eval_val_sin_placeholders("w2v_domain")

## ELMO

In [ ]:
# Experimento 4: ELMo (embeddings CONTEXTUALES de 1024 dim: el vector de una palabra depende de su frase).
# No hay matriz token -> vector, así que no se usa la capa Embedding: la CNN recibe directamente los vectores.
#
# OJO, ELMo es muy lento en CPU (~3 s por prompt de 256 tokens en este portátil; ~13.700 prompts en total).
# Por eso (1) los vectores se calculan UNA sola vez y se guardan en disco (float16),
#          (2) el cálculo se puede interrumpir y reanudar, y
#          (3) se usa un truncado menor (args.elmo_max_tokens) que en los experimentos de Word2Vec.
# Con GPU / Colab es mucho más rápido.
os.environ["TF_USE_LEGACY_KERAS"] = "1"
from simple_elmo import ElmoModel

ELMO_DIM = 1024

elmo = ElmoModel()
elmo.load(args.elmo_dir)


def build_elmo_cache(token_lists, path, max_len, batch_size=16):
    """Calcula los vectores ELMo de cada prompt (truncado cabeza + cola a max_len) y los guarda en un
    .npy float16 de forma (n_prompts, max_len, 1024). Reanudable: el progreso se guarda en un .progress."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    progress_file = path.with_suffix(".progress")
    n = len(token_lists)

    done = 0
    mm = None
    if path.exists() and progress_file.exists():
        mm = np.lib.format.open_memmap(path, mode="r+")
        if mm.shape == (n, max_len, ELMO_DIM):
            done = int(progress_file.read_text())
        else:
            mm = None
    if mm is None:
        mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16,
                                       shape=(n, max_len, ELMO_DIM))

    for start in range(done, n, batch_size):
        batch = [truncate_head_tail(t, max_len) or ["."] for t in token_lists[start:start + batch_size]]
        vecs = elmo.get_elmo_vectors(batch)          # (b, longitud del prompt más largo del batch, 1024)
        for i, tokens in enumerate(batch):
            mm[start + i, :len(tokens)] = vecs[i, :len(tokens)]   # el resto (padding) queda a cero
        mm.flush()
        progress_file.write_text(str(start + len(batch)))
        print(f"\rELMo {path.name}: {start + len(batch)}/{n}", end="")
    print()
    return path


class ElmoDataset(Dataset):
    """Misma interfaz que PromptDataset, pero la entrada son los vectores ELMo guardados en disco."""
    def __init__(self, caches, labels, pos_weight):
        self._data = {s: (np.load(caches[s], mmap_mode="r"), torch.from_numpy(labels[s]))
                      for s in caches}
        self.pos_weight = pos_weight
        self.set_split("train")

    def set_split(self, split="train"):
        self._X, self._y = self._data[split]

    def __len__(self):
        return len(self._y)

    def __getitem__(self, index):
        return {'x_data': torch.from_numpy(np.asarray(self._X[index], dtype=np.float32)),
                'y_target': self._y[index]}


elmo_caches, elmo_labels = {}, {}
for split in ["train", "val", "test"]:
    sub = df[df.split == split]
    elmo_caches[split] = build_elmo_cache(
        sub.tokens_flat.tolist(),
        Path(args.elmo_cache_dir) / f"{split}_{args.elmo_max_tokens}.npy",
        args.elmo_max_tokens)
    elmo_labels[split] = sub.jailbreak.to_numpy(dtype=np.float32)

elmo_dataset = ElmoDataset(elmo_caches, elmo_labels, pos_weight=dataset.pos_weight)

result_elmo = run_experiment("elmo", elmo_dataset, args,
                             use_embedding_layer=False, embed_dim=ELMO_DIM)

In [ ]:
# Validación SIN prompts con [PROMPT] / [TARGETLANGUAGE] (mejor checkpoint de este experimento)
df_val_sin_ph_elmo = eval_val_sin_placeholders("elmo")

## Comparación de modelos (validación)
Cada experimento con su mejor época y el umbral elegido en validación. **El test no se usa aquí**: la elección del modelo se hace solo con estas métricas de validación.

In [ ]:
# Tabla comparativa con las métricas de VALIDACIÓN (un experimento por fila), ordenada por MCC.
# Para ver las curvas de loss y MCC (train / val) de cada experimento en TensorBoard:
#   desde la terminal:  tensorboard --logdir runs        (en la carpeta notebooks)
#   o dentro del notebook:  %load_ext tensorboard   y luego   %tensorboard --logdir runs
rows = []
for name, r in all_results.items():
    rows.append({"Experimento": name,
                 "Mejor época": r["best_epoch"],
                 "Umbral": r["threshold"],
                 **r["val_metrics"]})

df_val = pd.DataFrame(rows)
if len(df_val):
    df_val = df_val.sort_values("MCC", ascending=False).reset_index(drop=True)
df_val.round(4)

## Test con el mejor modelo

In [ ]:
# Test: se evalúa UNA SOLA VEZ, con el experimento que hayas elegido mirando la tabla de validación.
# Escribe aquí su nombre: "w2v_raw", "w2v_finetuned", "w2v_domain" o "elmo"
EXPERIMENTO_ELEGIDO = None

if EXPERIMENTO_ELEGIDO is None:
    print("Elige el experimento mirando la tabla de validación y escribe su nombre en EXPERIMENTO_ELEGIDO.")
else:
    r = all_results[EXPERIMENTO_ELEGIDO]
    # Mejor checkpoint de ese experimento y el umbral elegido en validación
    test_loss, y_test, p_test = evaluate(r["model"], r["dataset"], "test", r["loss_func"],
                                         args.batch_size, args.device)
    test_metrics = compute_metrics(y_test, p_test, r["threshold"])

    df_test = pd.DataFrame([{"Experimento": EXPERIMENTO_ELEGIDO,
                             "Umbral (elegido en val)": r["threshold"],
                             **test_metrics}])
    display(df_test.round(4))

    # ----- Guardado PERSISTENTE del resultado de test (para no tener que volver a ejecutarlo) -----
    # - test_<exp>.json: umbral, métricas de test y de validación
    # - test_<exp>_predicciones.npz: etiquetas y probabilidades de cada prompt de test
    #   (permite recalcular cualquier métrica o subconjunto sin volver a pasar el modelo)
    out_dir = Path(args.model_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"test_{EXPERIMENTO_ELEGIDO}.json", "w", encoding="utf-8") as f:
        json.dump({"experimento": EXPERIMENTO_ELEGIDO,
                   "umbral": float(r["threshold"]),
                   "mejor_epoca": int(r["best_epoch"]),
                   "n_prompts_test": int(len(y_test)),
                   "n_jailbreaks_test": int(y_test.sum()),
                   "test_loss": float(test_loss),
                   "metricas_test": {k: float(v) for k, v in test_metrics.items()},
                   "metricas_val": {k: float(v) for k, v in r["val_metrics"].items()}},
                  f, indent=2, ensure_ascii=False)
    np.savez(out_dir / f"test_{EXPERIMENTO_ELEGIDO}_predicciones.npz",
             y_true=y_test, y_prob=p_test)
    print(f"Resultados de test guardados en {out_dir}/test_{EXPERIMENTO_ELEGIDO}.json "
          f"y test_{EXPERIMENTO_ELEGIDO}_predicciones.npz")

    # ----- Guardado también en TensorBoard (en el log del experimento elegido) -----
    log_to_tensorboard(
        EXPERIMENTO_ELEGIDO, "test",
        {**test_metrics, "loss": test_loss, "umbral": r["threshold"]},
        table_text="    " + df_test.round(4).to_string(index=False).replace("\n", "\n    "),
        step=r["best_epoch"])
    print("Resultados de test añadidos al log de TensorBoard de", EXPERIMENTO_ELEGIDO)